# 긴 대화 요약: SummarizationMiddleware

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationSummaryMemory`와 `ConversationSummaryBufferMemory` 대신,
LangChain v1의 agent에는 `SummarizationMiddleware`를 적용할 수 있습니다. 임계치에
도달하면 오래된 메시지를 요약으로 대체하고 최근 메시지는 그대로 유지합니다.

단순 trimming과 달리 오래된 정보의 핵심을 남길 수 있지만, 요약용 모델 호출 비용과
요약 오류 가능성을 함께 고려해야 합니다.


In [ ]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.4-mini")
model = init_chat_model(MODEL_ID)


## 요약 미들웨어가 있는 agent

실습에서는 동작을 빨리 보기 위해 메시지 개수 기준을 작게 잡습니다. 운영에서는 보통
`trigger=("tokens", 4000)`처럼 실제 모델의 컨텍스트 예산에 맞춘 토큰 기준을 사용합니다.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

summary_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "당신은 여행 상품 상담원입니다. 가격·포함 사항·취소 조건을 정확히 기억하고 "
        "모르는 내용은 추측하지 마세요."
    ),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 8),
            keep=("messages", 4),
        )
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "europe-package"}}


In [ ]:
questions = [
    "유럽 14박 15일 패키지의 기본 가격은 3,500유로라고 기억해 주세요.",
    "주요 방문지는 파리, 로마, 베를린, 취리히입니다.",
    "기본 여행자 보험이 포함되고, 비즈니스석 업그레이드는 1,200유로입니다.",
    "호텔은 4성급이며 매일 조식만 포함됩니다.",
    "예약금은 500유로이고 30일 전까지 취소하면 전액 환불됩니다.",
    "지금까지 알려드린 가격과 취소 조건을 정리해 주세요.",
]

for question in questions:
    result = summary_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    print(f"Q: {question}\nA: {result['messages'][-1].content}\n")


## 요약 후 checkpoint 상태 확인

임계치를 넘으면 오래된 원문 메시지 일부가 요약 컨텍스트로 대체됩니다. 정확한 내부
메시지 모양은 버전에 따라 달라질 수 있으므로, 특정 문자열 형식이 아니라 의미와
`messages` 상태를 기준으로 다룹니다.


In [ ]:
messages = summary_agent.get_state(config).values["messages"]

for index, message in enumerate(messages, start=1):
    preview = str(message.content).replace("\n", " ")[:240]
    print(f"{index:02d}. [{message.type}] {preview}")


요약은 손실 압축입니다. 반드시 보존해야 하는 주문 번호, 금액, 사용자 설정 같은 값은
요약문에만 맡기지 말고 별도의 구조화 상태나 장기 메모리 Store에 저장하세요.
